# Lab 2 · Python thuần trên dữ liệu thật

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành**  

Giữ bài làm trong notebook này. Với Colab: **File → Save a copy in Drive** trước khi sửa.
Khi chuyển sang GitHub, nộp đúng file `lab-02.ipynb`; lưu trên Drive chưa phải nộp bài.

## Mục tiêu

1. Đọc CSV bằng `csv.DictReader`, nhận biết dữ liệu đọc lên là chuỗi.
2. Làm sạch giá bằng `try/except`, giữ đúng khác biệt giữa thiếu và 0.
3. Tổng hợp bằng list, comprehension, dict, set và hàm.
4. So sánh mean/median, kiểm tra chéo cột và xuất JSON.

## Cách làm việc

- Khởi động và bài có hướng dẫn: tự gõ, **không dùng AI**, theo chính sách môn học.
- Phần **Bài tự làm ✅ mở**: được dùng AI, phải ghi prompt, điều đã kiểm chứng và giới hạn.
- Điền cell **Bài làm** và **Trả lời**. Giữ tên hàm/biến, mã câu và các cell công khai đã cho.
- Public checks chỉ cho biết ví dụ nhỏ đã qua. Grader sẽ dùng dữ liệu khác cùng yêu cầu công khai.
- Cell chưa làm được báo `CHƯA LÀM`; một public check lỗi không dừng các check khác.
  Cơ chế này không che lỗi cú pháp ở cell bài làm: bạn vẫn cần sửa lỗi cú pháp để Run all được.
- **Restart & Run all** trước khi nộp. Việc notebook chạy hết với TODO chưa hoàn thành không có nghĩa bài đã đạt.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Q1–Q8: tám kỹ năng, mỗi câu 10 điểm | 80 | Test tự động; giảng viên kiểm tra yêu cầu Python thuần |
| Diễn giải và bằng chứng trên dữ liệu thực hành | 15 | Giảng viên |
| Khai báo AI / kiểm chứng phần mở | 5 | Giảng viên |

Chỉ dùng thư viện chuẩn Python trong Q1–Q8; không dùng pandas/NumPy.
Đầu vào không được sửa tại chỗ trừ khi đề nói rõ. Hàm phải nhận dữ liệu qua tham số.

Điểm được tính theo từng mục, không theo số lượng public checks. Phần diễn giải do giảng viên đọc.
Các câu độc lập dự kiến được chấm với đầu vào riêng; không trừ toàn bộ lab vì một hàm trước đó sai.


In [ ]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")


## Phần 0 · Khởi động có hướng dẫn — khoảng 10 phút

Tự gõ ba ví dụ, thay một đầu vào rồi dự đoán kết quả trước khi chạy.
Đây là ví dụ học, không phải bài nộp tính điểm.


In [ ]:
gia_tb, ty_le = 118199.6, 0.178
print(f"Giá TB: {gia_tb:,.0f} CLP | Tỷ lệ: {ty_le:.1%}")
gia_mau = [52000, None, 78000, 0]
gia_co = [g for g in gia_mau if g is not None]
print("Giá hợp lệ:", gia_co, "Trung bình:", sum(gia_co) / len(gia_co))
thang_3, thang_6 = {101, 102, 103, 104}, {102, 104, 105}
print("Biến mất:", thang_3 - thang_6, "Mới thêm:", thang_6 - thang_3)


Giá TB: 118,200 CLP | Tỷ lệ: 17.8%
Giá hợp lệ: [52000, 78000, 0] Trung bình: 43333.333333333336
Biến mất: {101, 103} Mới thêm: {105}


## Dữ liệu: thử nhanh và thực hành trên snapshot thật

**Bản nháp mở sẵn bằng 6 dòng giả lập**, chứa giá thiếu và giá bằng 0 để thử quy tắc xử lý.
Không dùng kết quả từ 6 dòng này để kết luận về Santiago.

Khi thực hành trên dữ liệu thật, lấy `listings.csv` bản **visualisations**, Santiago,
snapshot **2026-06-29**, giá **CLP/đêm**, từ [Inside Airbnb](https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations/listings.csv),
hoặc bản sao cùng snapshot do giảng viên cung cấp. Đưa file vào Colab/Jupyter rồi điền
`LISTINGS_PATH` trong cell dưới. Nếu nguồn tải không hoạt động, không thay bằng snapshot khác
mà không ghi rõ. Đường dẫn là cấu hình dữ liệu thực hành, không phải câu trả lời.

Public checks dùng các bộ dữ liệu nhỏ riêng, nên vẫn chạy được khi chưa có snapshot thật.
Các con số của snapshot gốc không được dùng làm hằng số trong lời giải.


In [ ]:
from pathlib import Path
import csv
import io

# Nếu đã có snapshot thật, điền đường dẫn, ví dụ "data/listings.csv".
# Để None khi xem bản nháp: dùng 6 dòng GIẢ LẬP ở dưới, không phải Santiago.
LISTINGS_PATH = '/content/data/listings (1).csv'
DEMO_CSV = 'id,name,neighbourhood,room_type,price,number_of_reviews,last_review\n1,Phong A,Centro,Entire home/apt,100,0,\n2,Phong B,Centro,Private room,50,2,2026-05-10\n3,Phong C,Norte,Entire home/apt,,0,\n4,Phong D,Norte,Entire home/apt,150,1,2026-06-01\n5,Phong E,Sur,Private room,0,3,2026-05-30\n6,Phong F,Sur,Entire home/apt,300,2,2026-06-10\n'

if LISTINGS_PATH is None:
    rows = list(csv.DictReader(io.StringIO(DEMO_CSV)))
    DATA_LABEL = "GIẢ LẬP — 6 dòng để thử bản nháp"
else:
    with open(LISTINGS_PATH, encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    DATA_LABEL = f"File người học cung cấp: {LISTINGS_PATH}"

print(DATA_LABEL)
print("Số dòng:", len(rows))
print(rows[:2])


File người học cung cấp: /content/data/listings (1).csv
Số dòng: 18534
[{'id': '978070332077815549', 'name': 'luminosa mansarda con balcón', 'host_id': '118157228', 'host_profile_id': '1468207531264648757', 'host_name': 'Patricia', 'neighbourhood_group': '', 'neighbourhood': 'Ñuñoa', 'latitude': '-33.43765', 'longitude': '-70.5833', 'room_type': 'Private room', 'price': '45647', 'minimum_nights': '4', 'number_of_reviews': '2', 'last_review': '2023-12-15', 'reviews_per_month': '0.06', 'calculated_host_listings_count': '5', 'availability_365': '269', 'number_of_reviews_ltm': '0', 'license': ''}, {'id': '1069858035768058539', 'name': 'Cómoda habitación bien ubicada con baño privado', 'host_id': '462786436', 'host_profile_id': '1470084981980135796', 'host_name': 'Luís Alfonso', 'neighbourhood_group': '', 'neighbourhood': 'Recoleta', 'latitude': '-33.42242', 'longitude': '-70.64116', 'room_type': 'Private room', 'price': '19856', 'minimum_nights': '1', 'number_of_reviews': '8', 'last_review

## Phần 1 · Xây các bước xử lý — khoảng 60 phút

Tình huống: công ty du lịch cần báo cáo giá và quy mô thị trường dưới dạng JSON.
Các câu được tách theo đầu vào để bạn vẫn làm được câu sau khi câu trước chưa xong.
Cuối phần có cell ghép chúng lại trên bảng thực hành.

CSV có các cột `id`, `name`, `neighbourhood`, `room_type`, `price`, `number_of_reviews`, `last_review`.
`price` có thể rỗng; số review trong CSV là chuỗi. Câu nào nhận **giá đã làm sạch** sẽ ghi rõ.


### Q1 · Đọc CSV — 10 điểm

Viết `read_listings(path)` trả về list các dict bằng `csv.DictReader`, dùng UTF-8.
Giữ giá trị dạng chuỗi, giữ thứ tự dòng; file chỉ có header trả `[]`.
Đề chấm file hợp lệ, không yêu cầu tự sửa CSV hỏng. Nhớ đóng file bằng `with`.


In [ ]:
def read_listings(path):
    with open(path, encoding="utf-8") as f:
      a = list(csv.DictReader(f))
    return a

In [ ]:
def check_q1_1():
    import tempfile
    from pathlib import Path
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "listings.csv"
        path.write_text("id,price,name\n1,0,Phòng nhỏ\n2,,Phòng lớn\n", encoding="utf-8")
        result = read_listings(path)
    assert result == [{"id": "1", "price": "0", "name": "Phòng nhỏ"}, {"id": "2", "price": "", "name": "Phòng lớn"}]

public_check("Q1 · ví dụ 1", check_q1_1)



[ĐẠT] Q1 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


### Q2 · Chuyển giá sang số — 10 điểm

Viết `to_float(value)`: chuỗi số hoặc số hữu hạn → `float`; `None`, chuỗi rỗng hoặc chuỗi không
chuyển được → `None`. Dùng `try/except` bắt `ValueError`, `TypeError`. Giữ giá bằng 0.
Đầu vào không chứa bool, NaN/Infinity, dấu tiền tệ hay dấu phân cách hàng nghìn;
không tự bổ sung quy tắc loại giá âm.


In [ ]:
def to_float(value):
    try:
      value = float(value)
    except ValueError:
      value = None
    except TypeError:
      value = None
    return value

In [ ]:
def check_q2_1():
    assert to_float("12.5") == 12.5
    assert to_float("0") == 0.0

public_check("Q2 · ví dụ 1", check_q2_1)


[ĐẠT] Q2 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [ ]:
def check_q2_2():
    assert to_float("") is None
    assert to_float(None) is None
    assert to_float("N/A") is None

public_check("Q2 · ví dụ 2", check_q2_2)


[ĐẠT] Q2 · ví dụ 2: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [ ]:
# Hàm được cung cấp, không phải bài cần tự cài lại.
def trung_vi(xs):
    ordered = sorted(xs)
    n = len(ordered)
    if n == 0:
        return None
    if n % 2:
        return ordered[n // 2]
    return (ordered[n // 2 - 1] + ordered[n // 2]) / 2


### Q3 · Tóm tắt cột giá đã làm sạch — 10 điểm

`summarize_prices(prices)` nhận list số hữu hạn hoặc `None`, trả dict có đúng các khóa
`n_valid`, `n_missing`, `mean`, `median`. Chỉ bỏ `None`, giữ 0; không có giá hợp lệ thì hai thống kê là `None`.
Không làm tròn trong hàm. Dùng `trung_vi` được cung cấp; câu này không phụ thuộc `to_float`.


In [ ]:
def summarize_prices(prices):
    if len(prices) == 0:
      return {"n_valid": 0, "n_missing": 0, "mean":None, "median":None}
    valid = 0
    trungvil = []
    for i in prices:
      if isinstance(i, float):
        valid += 1
        trungvil.append(i)

    return {"n_valid": valid, "n_missing": len(prices)-valid, "mean":sum(trungvil)/len(trungvil), "median":trung_vi(trungvil)}

In [ ]:
def check_q3_1():
    result = summarize_prices([10.0, None, 0.0, 20.0])
    assert result == {"n_valid": 3, "n_missing": 1, "mean": 10.0, "median": 10.0}

public_check("Q3 · ví dụ 1", check_q3_1)


[ĐẠT] Q3 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [ ]:
def check_q3_2():
    assert summarize_prices([]) == {"n_valid": 0, "n_missing": 0, "mean": None, "median": None}

public_check("Q3 · ví dụ 2", check_q3_2)


[ĐẠT] Q3 · ví dụ 2: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


### Q4 · Tìm phòng đắt nhất — 10 điểm

`most_expensive(records)` nhận list dict có `id` dạng chuỗi và `price` đã là số hữu hạn hoặc `None`.
Trả `id` phòng đắt nhất bằng `max(..., key=...)`; bỏ giá thiếu, giữ giá 0.
Nếu đồng giá cao nhất, lấy dòng xuất hiện đầu tiên; nếu không có giá hợp lệ trả `None`.


In [ ]:
def most_expensive(records):
  for i in records:
    if i["price"] == None:
      i["price"] = 0
  return max(records, key = lambda a: a["price"])["id"]

In [ ]:
def check_q4_1():
    data = [{"id": "a", "price": None}, {"id": "b", "price": 25.5}, {"id": "c", "price": 10.0}]
    assert most_expensive(data) == "b"

public_check("Q4 · ví dụ 1", check_q4_1)


[ĐẠT] Q4 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


### Q5 · Đếm phòng theo khu — 10 điểm

`count_by_area(records)` nhận list dict có `neighbourhood` là chuỗi không rỗng,
trả dict `{tên_khu: số_dòng}` bằng dict cộng dồn. Đếm mọi dòng, không phụ thuộc giá.
List rỗng trả `{}`. Không yêu cầu thứ tự khóa.


In [ ]:
def count_by_area(records):
    neibourdict = {}
    for i in records:
      if i["neighbourhood"] in neibourdict:
        neibourdict[i["neighbourhood"]] += 1
      else:
        neibourdict[i["neighbourhood"]] = 1
    return neibourdict


In [ ]:
def check_q5_1():
    assert count_by_area([{"neighbourhood": "Ñuñoa"}, {"neighbourhood": "Centro"}, {"neighbourhood": "Ñuñoa"}]) == {"Ñuñoa": 2, "Centro": 1}

public_check("Q5 · ví dụ 1", check_q5_1)


[ĐẠT] Q5 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


### Q6 · Giá trung vị theo khu — 10 điểm

`median_by_area(records)` nhận list dict có `neighbourhood` không rỗng và `price` đã là số hoặc `None`.
Trả dict cho **mọi khu xuất hiện**; khu không có giá hợp lệ nhận `None`.
List rỗng trả `{}`. Dùng `trung_vi`; không chỉ tính một danh sách khu cố định.


In [ ]:
def median_by_area(records):
    if len(records) == 0:
      return {}
    nei = {}
    for i in records:
      if i["neighbourhood"] in nei:
        nei[i["neighbourhood"]] += 1
      elif i["price"] == None:
        nei[i["neighbourhood"]] = None
      else:
        nei[i["neighbourhood"]] = 1
    for key, value in nei.items():
      if value != 1 and value != None:
        pr = []
        for j in records:
          if j["neighbourhood"] == key:
            pr.append(j["price"])

        nei[key] = trung_vi(pr)
    return nei
print(median_by_area([{"neighbourhood": "A", "price": 10.0}, {"neighbourhood": "A", "price": 30.0}, {"neighbourhood": "B", "price": None}]))

{'A': 20.0, 'B': None}


In [ ]:
def check_q6_1():
    data = [{"neighbourhood": "A", "price": 10.0}, {"neighbourhood": "A", "price": 30.0}, {"neighbourhood": "B", "price": None}]
    assert median_by_area(data) == {"A": 20.0, "B": None}

public_check("Q6 · ví dụ 1", check_q6_1)


[ĐẠT] Q6 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


### Q7 · Phòng chưa review và kiểm tra chéo — 10 điểm

`review_qa(records)` nhận các dòng CSV có `number_of_reviews` là chuỗi biểu diễn số nguyên không âm,
`last_review` là chuỗi ngày hoặc chuỗi rỗng. Trả dict `n_zero`, `n_inconsistent`.
`n_zero`: số dòng có 0 review. `n_inconsistent`: **trong các dòng đó**, số dòng có ngày review khác rỗng.
Không tự sửa dữ liệu; list rỗng trả hai số 0.


In [ ]:
def review_qa(records):
  a = 0
  b = 0
  if len(records) == 0:
    return {"n_zero": 0, "n_inconsistent": 0}
  for i in records:
    if i["number_of_reviews"] == "0":
      a += 1
    if not i["last_review"]:
      b += 1
  return {"n_zero": a, "n_inconsistent":b}


In [ ]:
def check_q7_1():
    data = [{"number_of_reviews": "0", "last_review": ""}, {"number_of_reviews": "0", "last_review": "2026-01-01"}, {"number_of_reviews": "2", "last_review": "2026-01-02"}]
    assert review_qa(data) == {"n_zero": 2, "n_inconsistent": 1}

public_check("Q7 · ví dụ 1", check_q7_1)


[ĐẠT] Q7 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


### Q8 · Ghi và đọc lại JSON — 10 điểm

`save_report(report, path)` ghi dict JSON UTF-8, dùng `ensure_ascii=False`, rồi đọc file bằng `json.load`
và trả dict đọc lại. `path` trỏ đến thư mục cha đã tồn tại, cho phép ghi đè.
Đầu vào chỉ gồm kiểu JSON chuẩn, số hữu hạn và `None`; không sửa dict đầu vào.


In [ ]:
import json

def save_report(report, path):
    with open(path, "w",encoding = "utf-8") as f:
      json.dump(report, f, ensure_ascii = False)
    with open(path, "r", encoding = "utf-8") as f:
      result = json.load(f)
    return result


In [ ]:
def check_q8_1():
    import tempfile
    from pathlib import Path
    report = {"thanh_pho": "Santiago", "gia": None, "khu": "Ñuñoa"}
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "report.json"
        result = save_report(report, path)
        assert path.exists(), "Cần thực sự ghi file."
        assert json.loads(path.read_text(encoding="utf-8")) == report
        assert result == report

public_check("Q8 · ví dụ 1", check_q8_1)


[ĐẠT] Q8 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


## Phần 2 · Ghép quy trình trên dữ liệu thực hành

Cell dưới dùng các hàm bạn vừa viết. Với bản nháp, báo cáo ghi rõ nguồn giả lập;
với snapshot thật, ghi nguồn file bạn đã chọn. Chưa được suy ra ý nghĩa của giá trị ngoại lai chỉ từ độ lớn.
Nếu một hàm còn TODO, phần ghép báo chưa hoàn thành nhưng các public checks độc lập vẫn chạy.


In [ ]:
def build_current_report():
    active_rows = read_listings(LISTINGS_PATH) if LISTINGS_PATH is not None else rows
    clean = [{**r, "price": to_float(r["price"])} for r in active_rows]
    counts = count_by_area(clean)
    top5 = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:5]
    print("Top khu theo số phòng:", top5)
    report = {"nguon": DATA_LABEL, "so_phong": len(clean),
              "gia": summarize_prices([r["price"] for r in clean]),
              "id_dat_nhat": most_expensive(clean),
              "gia_trung_vi_theo_khu": median_by_area(clean),
              "review_qa": review_qa(active_rows)}
    return save_report(report, "bao_cao_lab02.json")

preview("Báo cáo dữ liệu thực hành", build_current_report)


Top khu theo số phòng: [('Santiago', 7182), ('Providencia', 2997), ('Las Condes', 2862), ('Ñuñoa', 1813), ('Lo Barnechea', 824)]
Báo cáo dữ liệu thực hành
{'nguon': 'File người học cung cấp: /content/data/listings (1).csv', 'so_phong': 18534, 'gia': {'n_valid': 17688, 'n_missing': 846, 'mean': 118199.6498190864, 'median': 59000.0}, 'id_dat_nhat': '651018255986860283', 'gia_trung_vi_theo_khu': {'Ñuñoa': 59378.0, 'Recoleta': 46720.0, 'Quilicura': 35000.0, 'Las Condes': 95000.0, 'Cerrillos': 45500.0, 'Maipú': 34903.0, 'La Florida': 50212.0, 'Pudahuel': 38344.0, 'Santiago': 46764.5, 'Peñalolén': 45929.5, 'Providencia': 70468.0, 'Independencia': 46275.0, 'Estación Central': 39922.5, 'Lo Barnechea': 414891.0, 'Quinta Normal': 36423.0, 'Lo Prado': 28351.5, 'Vitacura': 117922.0, 'La Reina': 45844.5, 'Macul': 44073.5, 'Huechuraba': 57059.0, 'La Pintana': 1, 'Cerro Navia': 36992.0, 'Renca': 29440.0, 'Pedro Aguirre Cerda': 22606.0, 'La Cisterna': 51923.5, 'El Bosque': 33882.5, 'Lo Espejo': 42597.

### Diễn giải và bằng chứng — 15 điểm

1. Mean và median trong dữ liệu bạn dùng khác nhau thế nào? Dẫn số liệu và giải thích vai trò của ngoại lai (5 điểm).
2. Vì sao lọc bằng `if price` có thể sai? Đưa một ví dụ bạn tự kiểm chứng (5 điểm).
3. Nếu dữ liệu có 0 review nhưng ngày review không rỗng, vì sao nên báo QA thay vì mặc định xóa dòng? (5 điểm).

Ghi rõ kết luận dựa trên dữ liệu giả lập hay Santiago thật.


### Trả lời câu hỏi diễn giải và bằng chứng (Dựa trên dữ liệu Santiago thật):

1. Mean và median trong dữ liệu em dùng khác nhau vì mean là trung bình cộng, median là trung vị, tính trung bình cộng là cộng tất cả các phòng rồi chia đều nên các giá trị cực đại có thể kéo lệch đồ thị về 1 phía, còn trung vị là giá trị đứng chính giữa danh sách đã sắp xếp nên phản ánh chính xác dữ liệu phổ biến
* **Mean (Trung bình cộng):** ~118,199.6 CLP/đêm
* **Median (Trung vị):** 59,000.0 CLP/đêm

  Để lọc đúng, ta phải so sánh tường minh: `if price is not None:`.
2. Vì trong python, giá trị số 0 hoặc 0.0 được coi giống như False, None hay chuỗi rỗng nên nếu ta dùng câu lệnh if price dể lọc hoặc kiểm tra giá trị hợp lệ thì những dữ liệu có giá thực tế bằng 0 sẽ bị lọc bỏ sai lệch
* **Ví dụ:**
  ```python
  price = 0.0
  if price:
      print("Giá hợp lệ")
  else:
      print("Bị coi là không hợp lệ hoặc thiếu")

3. Việc xoá dòng khi có 0 reivew nhưng ngày review khác rỗng có thể làm mất dữ liệu gốc, có thể làm mất các thông tin khác của dòng đó. Hoặc có thể do hệ thống gặp lỗi khi cập nhật số lượng review. Báo cho bộ phận QA giúp đội ngũ kỹ thuật tìm ra lỗi của hệ thống thay vì xoá thẳng tay

## Bài tự làm ✅ mở — mở rộng, chưa cộng vào 80 điểm tự động

**E1. Xếp hạng khu:** xây bảng cho mọi khu có ít nhất `min_valid` giá hợp lệ; cột tên khu, số giá hợp lệ,
trung vị; xếp trung vị giảm dần, đồng hạng thì tên khu tăng dần. Dùng `min_valid=100` với Santiago,
ngưỡng nhỏ hơn khi tự thử. In bằng f-string; không dùng số khu hay vị trí đứng đầu có sẵn làm đáp án.

**E2. Review theo năm, nâng cao:** với [reviews.csv cùng snapshot](https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations/reviews.csv),
đọc bằng `csv.DictReader`, đếm `date[:4]` bằng dict. Nêu giới hạn khi so một năm chưa đủ với năm đầy đủ;
không khẳng định nguyên nhân biến động chỉ từ số đếm.

Bạn được dùng AI ở phần này. Mục khai báo/kiểm chứng cuối notebook chiếm 5 điểm; nếu không dùng, ghi rõ
và đưa một phép kiểm bạn tự thiết kế. Không trừ điểm vì không dùng AI.


### E1 · Xếp hạng khu theo trung vị giá

**Hàm:** `rank_areas(records: list[dict], min_valid: int = 100) -> list[dict]`.

- Mỗi dòng có `neighbourhood` là chuỗi không rỗng và `price` **đã làm sạch** thành số hữu hạn hoặc `None`.
  Dùng đầu vào này để thử độc lập với Q2; khi chạy trên CSV thật, chuyển giá trước khi gọi.
- `min_valid` là số nguyên dương. Chỉ giữ khu có ít nhất `min_valid` giá khác `None`; giữ giá 0.
- Trả list dict có đúng các khóa `neighbourhood` (str), `n_valid` (int), `median_price` (float).
  Xếp `median_price` giảm dần; đồng hạng xếp tên khu tăng dần theo thứ tự chuỗi Python.
- Không có khu đạt ngưỡng thì trả `[]`. Không làm tròn median; không sửa input.
- Chỉ dùng Python chuẩn. Bạn được tái dùng `trung_vi` được cung cấp; không gọi các hàm TODO khác để chấm E1.
- Hàm chỉ trả dữ liệu; phần in bảng bằng f-string nằm ở cell gọi hàm bên dưới.

E1 vẫn là phần mở rộng, chưa cộng vào 80 điểm tự động của bản nháp.


In [ ]:
def rank_areas(records: list[dict], min_valid: int = 100) -> list[dict]:
    """Xếp khu theo median giảm dần; hòa thì tên tăng dần.

    Trả [{"neighbourhood": str, "n_valid": int, "median_price": float}, ...].
    records nhận price đã là số hoặc None; không sửa input.
    """
    # TODO
    raise NotImplementedError


In [ ]:
def check_e1_1():
    from copy import deepcopy
    sample = [{"neighbourhood": "B", "price": 10.0}, {"neighbourhood": "B", "price": 30.0},
              {"neighbourhood": "A", "price": 20.0}, {"neighbourhood": "A", "price": 20.0},
              {"neighbourhood": "C", "price": None}]
    before = deepcopy(sample)
    result = rank_areas(sample, min_valid=2)
    assert result == [{"neighbourhood": "A", "n_valid": 2, "median_price": 20.0},
                      {"neighbourhood": "B", "n_valid": 2, "median_price": 20.0}]
    assert all(type(r["n_valid"]) is int and type(r["median_price"]) is float for r in result)
    assert sample == before

public_check("E1 · mở rộng · ví dụ 1", check_e1_1)


[CHƯA LÀM] E1 · mở rộng · ví dụ 1: Điền phần TODO rồi chạy lại.


In [ ]:
def check_e1_2():
    assert rank_areas([], min_valid=1) == []
    assert rank_areas([{"neighbourhood": "A", "price": 0.0}, {"neighbourhood": "A", "price": None}], 1) == [
        {"neighbourhood": "A", "n_valid": 1, "median_price": 0.0}]

public_check("E1 · mở rộng · ví dụ 2", check_e1_2)


[CHƯA LÀM] E1 · mở rộng · ví dụ 2: Điền phần TODO rồi chạy lại.


In [ ]:
def show_area_ranking():
    # Ghép với Q2 trên dữ liệu thực hành; public checks của E1 không phụ thuộc Q2.
    clean = [{**r, "price": to_float(r["price"])} for r in rows]
    threshold = 1 if LISTINGS_PATH is None else 100
    result = rank_areas(clean, min_valid=threshold)
    for r in result:
        print(f"{r['neighbourhood']:<20}{r['n_valid']:>10,d}{r['median_price']:>14,.1f}")
    return f"{len(result)} khu đạt ngưỡng {threshold}; nguồn: {DATA_LABEL}"

preview("Bảng xếp hạng khu", show_area_ranking)


Bảng xếp hạng khu: chưa chạy được vì còn TODO.


### E2 · Đếm review theo năm

**Hàm:** `review_counts_by_year(records: list[dict]) -> dict[str, int]`.

- Input là list dòng từ `csv.DictReader`; mỗi dòng có `date` là ngày hợp lệ định dạng `YYYY-MM-DD`
  hoặc chuỗi rỗng. Các cột khác như `listing_id` không ảnh hưởng phép đếm.
- Bỏ dòng có `date == ""`; mỗi dòng còn lại tính là một review, **không tự loại trùng**.
- Trả dict `{năm_dạng_chuỗi: số_review}`, chỉ chứa năm có review, thứ tự khóa năm tăng dần.
  Dữ liệu rỗng hoặc toàn ngày rỗng trả `{}`. Không sửa input.
- Dùng Python chuẩn: lát cắt `date[:4]` và dict cộng dồn. Không cần tự kiểm tra ngày hỏng ngoài hợp đồng này.
- Không tải file trong hàm. Cell đọc CSV bên dưới được cung cấp để thử trên snapshot thật.

E2 là phần nâng cao, chưa có điểm tự động trong bản nháp. Kết quả số đếm không tự chứng minh nguyên nhân biến động.


In [ ]:
def review_counts_by_year(records: list[dict]) -> dict[str, int]:
    """Đếm từng dòng có date không rỗng, trả dict năm tăng dần.

    date có định dạng YYYY-MM-DD hoặc ""; không tự loại dòng trùng.
    """
    # TODO
    raise NotImplementedError


In [ ]:
def check_e2_1():
    from copy import deepcopy
    sample = [{"listing_id": "a", "date": "2026-01-01"},
              {"listing_id": "b", "date": "2025-12-31"},
              {"listing_id": "a", "date": "2026-01-01"},
              {"listing_id": "c", "date": ""}]
    before = deepcopy(sample)
    result = review_counts_by_year(sample)
    assert result == {"2025": 1, "2026": 2}
    assert list(result) == ["2025", "2026"]
    assert all(type(v) is int for v in result.values())
    assert sample == before

public_check("E2 · mở rộng · ví dụ 1", check_e2_1)


[CHƯA LÀM] E2 · mở rộng · ví dụ 1: Điền phần TODO rồi chạy lại.


In [ ]:
def check_e2_2():
    assert review_counts_by_year([]) == {}
    assert review_counts_by_year([{"date": ""}]) == {}

public_check("E2 · mở rộng · ví dụ 2", check_e2_2)


[CHƯA LÀM] E2 · mở rộng · ví dụ 2: Điền phần TODO rồi chạy lại.


In [ ]:
# File reviews.csv cùng snapshot, nếu đã tải được. None để thử bằng 3 dòng giả lập.
REVIEWS_PATH = None
if REVIEWS_PATH is None:
    review_rows = [{"listing_id": "1", "date": "2025-01-01"},
                   {"listing_id": "2", "date": "2026-01-02"},
                   {"listing_id": "1", "date": "2026-02-03"}]
    review_source = "GIẢ LẬP — không dùng để kết luận về Santiago"
else:
    with open(REVIEWS_PATH, encoding="utf-8", newline="") as f:
        review_rows = list(csv.DictReader(f))
    review_source = f"File đã chọn: {REVIEWS_PATH}"

preview(review_source, lambda: review_counts_by_year(review_rows))


GIẢ LẬP — không dùng để kết luận về Santiago: chưa chạy được vì còn TODO.


### Nhận xét phần mở rộng

- E1: nguồn dữ liệu, ngưỡng đã dùng, khu đứng đầu và một phép kiểm tay: [Điền]
- E2: nguồn dữ liệu, năm có nhiều review nhất và phạm vi thời gian của dữ liệu: [Điền]
- Có thể so số đếm năm 2026 chưa đầy đủ với toàn năm 2025 để kết luận thị trường co lại không? Vì sao? [Điền]
- Khai báo AI ở mục cuối notebook. Nếu không làm phần tùy chọn, ghi “Chưa làm phần mở rộng”.


## Trước khi nộp

1. Điền tên, mã sinh viên và nguồn dữ liệu đã dùng ở cell dưới.
2. Restart & Run all; sửa các lỗi và đọc lại bảng public checks.
3. Hoàn thành câu trả lời diễn giải và khai báo AI, kể cả khi không dùng AI.
4. Giữ tên file và cell bài làm. Không chép kết quả hiển thị thành hằng số thay cho phép xử lý.
5. Khi giảng viên phát hành hướng dẫn nộp chính thức: lưu notebook vào repo cá nhân của môn,
   commit/push và tạo tag theo hướng dẫn của lab. **Bản nháp này chưa nối với grader đang chạy.**


**Họ tên / mã sinh viên:** Phạm Quốc Đạt / 25022820

**Nguồn dữ liệu thực hành:**  Santiago 2026-06-29

**Khai báo AI cho phần được phép dùng:**

- Công cụ/model, hoặc “Không dùng”: Gemini 3.5 Flash
- Prompt chính: Hỏi về cú pháp Python là chính, cách sử dụng json và path, open with(), ...
- Tôi đã tự dự đoán gì trước khi hỏi: chưa có dự đoán
- Kết quả AI cần sửa hoặc điểm tôi chưa chắc: không có
- Cách tôi kiểm chứng, kèm ví dụ cụ thể: không có


In [ ]:
show_public_summary()


PUBLIC CHECKS — không phải điểm chính thức
Q1 · ví dụ 1: ĐẠT
Q2 · ví dụ 1: ĐẠT
Q2 · ví dụ 2: ĐẠT
Q3 · ví dụ 1: ĐẠT
Q3 · ví dụ 2: ĐẠT
Q4 · ví dụ 1: ĐẠT
Q5 · ví dụ 1: ĐẠT
Q6 · ví dụ 1: ĐẠT
Q7 · ví dụ 1: ĐẠT
Q8 · ví dụ 1: ĐẠT
E1 · mở rộng · ví dụ 1: CHƯA LÀM
E1 · mở rộng · ví dụ 2: CHƯA LÀM
E2 · mở rộng · ví dụ 1: CHƯA LÀM
E2 · mở rộng · ví dụ 2: CHƯA LÀM
Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.
